In [ ]:
!nvidia-smi

Wed Nov  4 16:49:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q ultralytics wandb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp /content/drive/MyDrive/kitti3.zip /content/
!unzip -q -o /content/kitti3.zip -d /content/
!ls /content/kitti3
!echo "train images:" && ls /content/kitti3/images/train | wc -l
!echo "val images:" && ls /content/kitti3/images/val | wc -l
!echo "test images:" && ls /content/kitti3/images/test | wc -l

images	kitti3.yaml  labels
train images:
5485
val images:
500
test images:
1496


In [ ]:
yaml_text = """path: /content/kitti3
train: images/train
val: images/val
test: images/test
names:
  0: vehicle
  1: pedestrian
  2: cyclist
"""
with open("/content/kitti3/data.yaml", "w") as f:
   f.write(yaml_text)
DATA = "/content/kitti3/data.yaml"
print(open(DATA).read())

path: /content/kitti3
train: images/train
val: images/val
test: images/test
names:
  0: vehicle
  1: pedestrian
  2: cyclist



In [ ]:
!pip install -q ultralytics wandb

from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/kitti3.zip /content/
!unzip -q -o /content/kitti3.zip -d /content/

yaml_text = """path: /content/kitti3
train: images/train
val: images/val
test: images/test
names:
  0: vehicle
  1: pedestrian
  2: cyclist
"""
with open("/content/kitti3/data.yaml", "w") as f:
  f.write(yaml_text)
DATA = "/content/kitti3/data.yaml"
print("Setup done:", DATA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup done: /content/kitti3/data.yaml


In [ ]:
import os
from google.colab import userdata

key = userdata.get("WANDB_API_KEY")   # stops with an error if the secret is missing
os.environ["WANDB_API_KEY"] = key

import wandb
wandb.login(key=key, relogin=True)

from ultralytics import settings
settings.update({"wandb": True})
print("W&B ready")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kaurharleet2408 (kaurharleet2408-personal-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
W&B ready


In [ ]:
from ultralytics import YOLO

RUNS = "/content/drive/MyDrive/driving-perception/runs"

baseline = YOLO("yolo11n.pt")  # smallest pretrained model
baseline.train(
    data=DATA,
    epochs=10,
    imgsz=640,
    batch=16,
    seed=42,
    project=RUNS,
    name="baseline_yolo11n",
    plots=True,
)

Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/kitti3/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_yolo11n-2, nbs=64, nms=None, opset=None, optimize=False

Using 5485 train, 500 val images for fraction=1.0 at imgsz=640
Using 2 dataloader workers
Logging results to /content/drive/MyDrive/driving-perception/runs/baseline_yolo11n-2
Starting training for 10 epochs...
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/10      2.42G       1.47      1.414      1.156         84        640: 100% ━━━━━━━━━━━━ 343/343 3.8it/s 1:31
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 4.1it/s 3.9s
                   all        500       2608      0.527      0.481      0.506      0.255

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10      2.42G       1.38     0.9679      1.117   

epoch,▁▂▂▃▄▅▅▆▇▇█▇
lr/pg0,▃▆█▇▆▅▄▃▂▁
lr/pg1,▃▆█▇▆▅▄▃▂▁
lr/pg2,▃▆█▇▆▅▄▃▂▁
metrics/mAP50(B),▁▁▂▄▄▆▆▆███
metrics/mAP50-95(B),▁▁▂▄▄▅▆▆███
metrics/precision(B),▁▇▂▅▇▅▅▆▇██
metrics/recall(B),▃▁▄▄▃▇█▅█▇▇
model/GFLOPs,▁
model/parameters,▁
+7,...


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b8e7289af30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [ ]:
improved = YOLO("yolo11s.pt")  # bigger model, more capacity
improved.train(
    data=DATA,
    epochs=30,
    imgsz=640,
    batch=16,
    seed=42,
    project=RUNS,
    name="improved_yolo11s",
    plots=True,
 )

Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/kitti3/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=improved_yolo11s, nbs=64, nms=None, opset=None, optimize=False, 

Using 5485 train, 500 val images for fraction=1.0 at imgsz=640
Using 2 dataloader workers
Logging results to /content/drive/MyDrive/driving-perception/runs/improved_yolo11s
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/30      4.08G      1.308     0.9853      1.077        195        640: 100% ━━━━━━━━━━━━ 343/343 3.7it/s 1:34
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 2.7it/s 5.9s
                   all        500       2608      0.833      0.438      0.578      0.312

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      4.11G      1.286      0.867      1.076        182        640: 100% ━━━━━━━━━━━━ 343/343 3.8it/s 1:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 4.4it/s 3.6s
                   all        500       2608      0.551

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇████
lr/pg0,▃▆██▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▁▁
lr/pg1,▃▆██▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▁▁
lr/pg2,▃▆██▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▁▁
metrics/mAP50(B),▂▁▂▂▃▄▅▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇███████
metrics/mAP50-95(B),▁▁▁▂▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██████
metrics/precision(B),▇▁▁▃▄▄▆▅▅▅▆▇▆▆▆▆▇▇▆▇▇▇▇▇▇████▇▇
metrics/recall(B),▁▃▄▃▄▅▆▅▅▆▆▅▆▇▇▇▇▇▇▇▇▇▇▇███████
model/GFLOPs,▁
model/parameters,▁
+7,...


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b8f46454670>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [ ]:
import json

def evaluate(run_name):
  weights = f"{RUNS}/{run_name}/weights/best.pt"
  model = YOLO(weights)
  m = model.val(data=DATA, split="test", project=RUNS, name=f"{run_name}_test")
  names = ["vehicle", "pedestrian", "cyclist"]
  out = {"model": run_name, "mAP50": float(m.box.map50), "mAP50-95": float(m.box.map)}
  for j, c in enumerate(m.box.ap_class_index):
    p, r, ap50, ap = m.box.class_result(j)
    out[names[int(c)]] = {
        "precision": float(p), "recall": float(r),
        "mAP50": float(ap50), "mAP50-95": float(ap),
    }
    return out
results = [evaluate("baseline_yolo11n"), evaluate("improved_yolo11s")]
with open(f"{RUNS}/test_results.json", "w") as f:
  json.dump(results, f, indent=2)
for r in results:
  print(r["model"], "| overall mAP50:", round(r["mAP50"], 3), "| mAP50-95:", round(r["mAP50-95"], 3))
  for cls in ("vehicle", "pedestrian", "cyclist"):
    if cls in r:
      print("   ", cls, "mAP50:", round(r[cls]["mAP50"], 3), "recall:", round(r[cls]["recall"], 3))


Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,537 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1731.5±520.3 MB/s, size: 49.6 KB)
val: Scanning /content/kitti3/labels/test... 1496 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1496/1496 1.8Kit/s 0.8s
val: New cache created: /content/kitti3/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 7.5it/s 12.6s
                   all       1496       7920      0.728      0.581      0.655      0.374
               vehicle       1347       6611      0.878      0.788      0.887      0.615
            pedestrian        381        957      0.637      0.514      0.584      0.276
               cyclist        252        352      0.668       0.44      0.493      0.231
Speed: 0.4ms preprocess, 2.2ms inference, 0.0ms loss, 1.5ms postprocess per image